# Week 2 Day 1 Lab - Evals and Observability

## Measuring an agent on UnderCover's harder claims

A new batch of 25 claims has arrived at UnderCover, and they're harder than the queue from Week 1. Many have long lists of items, with payments from airlines, hotels and other insurers to take off, and deductibles, limits and caps to apply.

We trace the agent from Week 1 in Langfuse and score its decisions against the handlers'. Then we run two experiments: the agent working out each payout itself, and the agent with a payout tool.

## 4 parts:

PART 1: The harder claims

PART 2: Tracing the agent with Langfuse

PART 3: Scoring an agent

PART 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week2day1free.ipynb">week2day1free.ipynb</a>.<br/><br/>
            You'll also need a free Langfuse account, which we set up below. Each experiment takes 2 or 3 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made in Week 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Connect to Langfuse

Langfuse is an open-source platform for tracing and evaluating LLM applications. You can host it yourself, or use Langfuse Cloud, which has a free Hobby plan.

1. Sign up at [cloud.langfuse.com](https://cloud.langfuse.com) and create a project
2. In the project's settings, go to API Keys and create a new key
3. Add the three values it shows to your `.env` file: `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_BASE_URL`

In [3]:
load_dotenv(override=True)
langfuse = get_client()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

Connected to Langfuse


## Part 1: The harder claims

The harder claims are in `data/evals/harder`, with their own policy records and the handlers' decisions. The policy wording is the same as before.

In [4]:
HARDER = Path("../data/evals/harder")


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

25 claims, 25 labels, 25 policies


In [5]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "items": len(claim["items"]),
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,items,amount_claimed,decision,payout
0,CLM-1201,Suitcase lost on the way to Lisbon,16,1725.77,approve,553.28
1,CLM-1202,Equipment case crushed in the hold on the way ...,9,7357.37,approve,2048.44
2,CLM-1203,Rented apartment in Rome broken into,14,6207.43,approve,2910.44
3,CLM-1204,Safari cut short when my sister was badly hurt...,11,5785.45,approve,4787.25
4,CLM-1205,Family of four held overnight at Newark by an ...,14,1681.14,approve,1021.14
5,CLM-1206,Golf clubs stolen from the trunk of a rental car,13,2587.96,approve,1450.01
6,CLM-1207,Backpack pulled off my chair at a cafe in Barc...,14,3581.17,approve,1306.72
7,CLM-1208,Torn knee ligament on the Routeburn Track,16,4518.80,approve,3668.80
8,CLM-1209,Broken wrist and a head wound when I fell out ...,16,5100.15,approve,4180.15
9,CLM-1210,Amalfi trip canceled after my cousin's heart a...,4,4796.70,decline,0.00


Similar mix of decisions

In [6]:
data["decision"].value_counts()

decision
approve    14
refer       6
decline     5
Name: count, dtype: int64

### A burglary in Rome

Here's one of the longer claims.

In [7]:
rome = [claim for claim in claims if claim["claim_id"] == "CLM-1203"][0]

print(rome["summary"])
print(rome["story"])
pd.DataFrame(rome["items"])

Rented apartment in Rome broken into
On July 22 Erik and I came back to our rented apartment in Trastevere after dinner to find the front door forced and our things gone. We called the police at once. The apartment owner's insurer has paid us $2,250. Receipts for everything over $100 are attached.


,description,category,amount
0,Laptop,electronics,1899.00
1,Tablet,electronics,649.00
2,Smartwatch,electronics,399.00
3,E-reader,electronics,159.99
4,Noise-canceling headphones,electronics,549.00
5,Handheld game console,electronics,599.99
6,VR headset,electronics,649.00
7,Portable speaker,electronics,189.95
8,Leather jacket,clothing,389.00
9,Two pairs of sneakers,clothing,258.00


This is how the handler worked out the payout.

In [8]:
decision = labels["CLM-1203"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

A locked apartment is not a place the public can get to, so the belongings were not unattended, and the police were called that night. Electronics Plus raises the limit to $1,200 for the laptop, and the tablet, the smartwatch and the e-reader are paid as claimed. Headphones, a game console and a VR headset are not on the Electronics Plus list and count at the $500 per-item limit on First. Cash is covered up to $100 in all. The items come to $5,160.44. The owner's insurer's $2,250 comes off, with no deductible on First. The payout of $2,910.44 is inside the $3,000 maximum and needs sign-off.


,step,amount
0,"Laptop, limited to $1,200.00",1200.00
1,Tablet,649.00
2,Smartwatch,399.00
3,E-reader,159.99
4,"Noise-canceling headphones, limited to $500.00",500.00
5,"Handheld game console, limited to $500.00",500.00
6,"VR headset, limited to $500.00",500.00
7,Portable speaker,189.95
8,Leather jacket,389.00
9,Two pairs of sneakers,258.00


## Part 2: Tracing the agent with Langfuse

This is the agent from Week 1 Day 2 with two changes. The customer's policy record goes in the prompt with the claim, so its only tool is `record_decision`. We also turn Luna's reasoning off with `reasoning_effort="none"`, which makes each call faster and cheaper.

In [9]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

The tool's docstring is in Google style. With `parse_docstring=True`, LangChain turns each line under `Args` into the description of that argument.

The rationale comes before the decision, so the model writes its reasons before it decides. As in UnderCover's claims system, a decision can't be changed once it's recorded, and an approval needs a payout above zero.

In [10]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="none", timeout=120, max_retries=5)

baseline_agent = create_agent(model=luna, tools=[record_decision], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

Langfuse's `CallbackHandler` sends a trace of everything LangChain does to Langfuse. We pass it to the agent in the `config`.

In [12]:
langfuse_handler = CallbackHandler()

recorded.clear()
result = baseline_agent.invoke(
    {"messages": [{"role": "user", "content": make_user_prompt(rome)}]},
    config={"callbacks": [langfuse_handler]},
)

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  record_decision (call_Xu6sYgKzQtGPIRW76UUXrKgI)
 Call ID: call_Xu6sYgKzQtGPIRW76UUXrKgI
  Args:
    claim_id: CLM-1203
    rationale: Approve the documented theft: it occurred during the covered trip, the police report was made promptly and lists the stolen property, and the receipts and insurer's payment letter are attached. Under sections 5 and 10, the First-plan per-item limit is $500, increased by Electronics Plus to $1,200 for qualifying electronics; the two pairs of sneakers count as one item, and cash is capped at $100 in total. Eligible value before the other insurer's payment is $5,659.93; subtract $2,250 paid by the apartment owner's insurer and the $0 First-plan deductible, then apply the $3,000 baggage maximum: payout $3,000.
    decision: approve
    payout: 3000
================================= Tool Message =================================
Name: record_decision

Recorded appro

Now open **Tracing** in Langfuse. The trace shows each call to the model with its prompt, its reply, the tokens and the time it took, and the tool call in between.

## Part 3: Scoring an agent

In Week 1 a claim was right when the decision matched the handler's and the payout was within a dollar. We still count that, but not every mistake costs UnderCover the same.

### The confusion matrix

A confusion matrix compares an agent's decisions with the right answers. Each row is the handler's decision and each column is the agent's. The claims on the diagonal are the ones where they agree, and each cell off the diagonal is a kind of mistake:

- **A wrongful decline:** the agent declines a claim the handler approved or referred. The customer is let down and may complain.
- **A wrongful payment:** the agent approves a claim the handler declined or referred. UnderCover pays money it doesn't owe.
- **A needless referral:** the agent refers a claim the handler decided. A handler has to spend time on it.

The confusion matrix compares decisions only, so an approval with the wrong payout still sits on the diagonal. The cost of mistakes below counts the wrong amounts.

### Precision and recall

Precision and recall come from the same matrix, with an approval as the positive:

- **Precision:** of the claims the agent approved, the share the handler approved.
- **Recall:** of the claims the handler approved, the share the agent approved.

### The cost of mistakes

A cost matrix gives each kind of mistake a cost in dollars. A wrongful payment costs the amount paid, and a wrong amount on a correct approval costs the difference.

In [13]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">The UnderCover business has more tolerance for false positives (we cover someone when we shouldn't) than false negatives (we decline someone when we should have covered them), because false negatives damage our reputation and customer loyalty. Our evals take this into account.<br/><br/>
            Agree the figures in the cost matrix with the business before you run your evals.
            </span>
        </td>
    </tr>
</table>

### Kappa

Accuracy can flatter an agent. 14 of the 25 claims are approvals, so an agent that approves everything gets 56% of the decisions right. Cohen's kappa corrects for this. It measures how much more often the agent agrees with the handlers than it would by chance, based on how often each of them picks each decision. A kappa of 1 is perfect agreement and 0 is no better than chance.

In [14]:
def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


approve_everything = pd.DataFrame({"handler": data["decision"], "agent": "approve"})

print(f"Approve everything gets a kappa of {kappa(approve_everything):.2f}")

Approve everything gets a kappa of 0.00


Here's the confusion matrix for approving everything. Every claim lands in the approve column.

In [15]:
def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()


show_confusion_matrix(approve_everything, "Approve everything")

### The straight-through rate

The straight-through rate is the share of claims the agent decides without referring them to a handler. It includes approvals over $1,000, which still need a handler's sign-off. An agent that refers every claim never pays or declines a claim wrongly. But its straight-through rate is 0%, and each of its 19 needless referrals costs $50. The handlers refer 6 of the 25 claims, so their rate is 76%.

## Part 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Datasets and experiments</h2>
            <span style="color:#118AB2;">A Langfuse dataset holds test inputs with their expected outputs. An experiment runs a task on each item of a dataset and scores the result with evaluators. An item evaluator scores one item and a run evaluator scores the whole run.<br/><br/>
            Langfuse keeps every run of a dataset with its scores and traces, so you can compare runs side by side.
            </span>
        </td>
    </tr>
</table>

### The dataset

We upload the harder claims as a dataset, with the handler's decision as the expected output. Each item's id is the claim id, so running this cell again updates the items in place.

In [16]:
DATASET = "harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]},
        metadata={"tiny": label["tiny"]},
    )

By default we run all 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

`N_TIMES` is the number of times each experiment runs. The agent's answers vary from run to run, and more runs give steadier scores. With more than one run, each confusion matrix adds up the claims from every run, and the comparison at the end averages the scores.

In [56]:
TINY = False
N_TIMES = 1

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

Running 25 of the 25 claims


### The task and the evaluators

The task runs an agent on one claim. If the agent finishes without recording a decision, the claim would go to a handler, so we count it as a referral.

In [57]:
def decide(agent, claim):
    """Run an agent on one claim and return its decision, with the cost of its model calls."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost}

`score_claim` is the item evaluator and `score_run` is the run evaluator. `score_table` puts each claim's answer beside the handler's.

In the run's scores, `decision_accuracy` is the share of claims on the diagonal of the confusion matrix. `accuracy` also needs the payout within a dollar.

In [58]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, and the cost of its mistake."""
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

### Experiment 1: the baseline

The agent works out each payout itself, following section 10 of the policy wording. Langfuse runs the claims one at a time, so each run takes about 2 minutes.

In [59]:
def baseline_task(*, item, **kwargs):
    """Run the baseline agent on one claim from the dataset."""
    return decide(baseline_agent, item.input)


baseline_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    baseline = langfuse.run_experiment(
        name="Baseline",
        description="The agent works out each payout itself",
        data=items,
        task=baseline_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(baseline.format())
    baseline_tables.append(score_table(baseline.item_results).assign(run=run))

CLM-1225: refer $0.00
CLM-1224: refer $0.00
CLM-1223: refer $0.00
CLM-1222: refer $0.00
CLM-1221: decline $0.00
CLM-1220: decline $0.00
CLM-1219: decline $0.00
CLM-1218: refer $0.00
CLM-1217: approve $1,336.90
CLM-1216: approve $3,000.00
CLM-1215: approve $1,500.00
CLM-1214: approve $622.60
CLM-1213: decline $0.00
CLM-1212: refer $0.00
CLM-1211: refer $0.00
CLM-1210: decline $0.00
CLM-1209: approve $4,179.65
CLM-1208: refer $0.00
CLM-1207: decline $0.00
CLM-1206: approve $1,500.00
CLM-1205: approve $1,012.14
CLM-1204: approve $4,726.75
CLM-1203: approve $3,000.00
CLM-1202: approve $2,789.99
CLM-1201: refer $0.00
Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Baseline
📋 Run name: Baseline - 2026-10-22T21:20:56.899183Z - The agent works out each payout itself
25 items
Evaluations:
  • mistake_cost
  • right

Average Scores:
  • mistake_cost: 160.337
  • right: 0.560

Run Evaluations:
  •

In [60]:
baseline_table = pd.concat(baseline_tables, ignore_index=True)
show_confusion_matrix(baseline_table, "Baseline")

Here are the claims it got wrong, with the agent's rationale.

In [61]:
baseline_table[~baseline_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,llm_cost,rationale,run
0,CLM-1201,approve,refer,553.28,0.00,False,50.00,0.000307,The carrier report confirms the suitcase was l...,1
1,CLM-1202,approve,approve,2048.44,2789.99,False,741.55,0.000332,Approve the covered physical carrier damage un...,1
2,CLM-1203,approve,approve,2910.44,3000.00,False,89.56,0.000347,Approve the documented theft under section 5: ...,1
3,CLM-1204,approve,approve,4787.25,4726.75,False,60.50,0.000308,Approve under section 2: interruption is cover...,1
4,CLM-1205,approve,approve,1021.14,1012.14,False,9.00,0.000312,The First plan covers a 27-hour carrier delay ...,1
5,CLM-1206,approve,approve,1450.01,1500.00,False,49.99,0.000333,Approve under Section 5: the theft was reporte...,1
6,CLM-1207,approve,decline,1306.72,0.00,False,1500.00,0.000289,Decline under section 5: the backpack was hang...,1
7,CLM-1208,approve,refer,3668.80,0.00,False,50.00,0.000365,Refer for review under sections 7 and 11. The ...,1
14,CLM-1215,approve,approve,1415.78,1500.00,False,84.22,0.000352,Approve the baggage-loss claim under section 5...,1
15,CLM-1216,approve,approve,1677.90,3000.00,False,1322.10,0.000289,The theft occurred during the covered trip and...,1


### Experiment 2: a payout tool

The payout rules in section 10 are fixed business logic, and business logic belongs in code. `week2day1_payout.py` has UnderCover's payout rules as a Python function, `work_out_payout`, and we give it to the agent as a tool. The agent still judges whether the claim is covered, which lines count and what counts as one item.

In [62]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

A tool is still a Python function, so we can try it ourselves. Here's a suitcase and three T-shirts on an Economy policy, where the airline has paid $100.

In [63]:
calculate_payout.invoke({
    "policy_id": "POL-2201",
    "benefit": "baggage",
    "lines": [
        {"description": "Hard-shell suitcase", "amount": 239.99, "kind": "item"},
        {"description": "Three T-shirts", "amount": 74.97, "quantity": 3, "kind": "item"},
    ],
    "paid_by_others": 100,
})

{'calculation': [{'step': 'Hard-shell suitcase, limited to $150.00',
   'amount': 150.0},
  {'step': 'Three T-shirts', 'amount': 74.97},
  {'step': 'Less what others have paid or refunded', 'amount': -100.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'payout': 24.97,
 'needs_sign_off': False}

The agent gets the tool and one more line in its instructions.

In [64]:
PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")


def payout_tool_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return decide(payout_agent, item.input)

In [65]:
payout_tool_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    with_payout_tool = langfuse.run_experiment(
        name="Payout tool",
        description="The agent works out each payout with the payout tool",
        data=items,
        task=payout_tool_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(with_payout_tool.format())
    payout_tool_tables.append(score_table(with_payout_tool.item_results).assign(run=run))

CLM-1225: refer $0.00
CLM-1224: approve $1,227.95
CLM-1223: refer $0.00
CLM-1222: refer $0.00
CLM-1221: decline $0.00
CLM-1220: decline $0.00
CLM-1219: decline $0.00
CLM-1218: decline $0.00
CLM-1217: approve $1,335.90
CLM-1216: approve $1,677.90
CLM-1215: approve $1,415.78
CLM-1214: approve $622.60
CLM-1213: decline $0.00
CLM-1212: refer $0.00
CLM-1211: refer $0.00
CLM-1210: decline $0.00
CLM-1209: decline $0.00
CLM-1208: approve $3,668.80
CLM-1207: approve $1,306.72
CLM-1206: approve $1,450.01
CLM-1205: approve $1,103.14
CLM-1204: approve $4,787.25
CLM-1203: approve $2,910.44
CLM-1202: approve $2,048.44
CLM-1201: approve $553.28
Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool
📋 Run name: Payout tool - 2026-10-22T21:23:28.336460Z - The agent works out each payout with the payout tool
25 items
Evaluations:
  • mistake_cost
  • right

Average Scores:
  • mistake_cost: 172.398


In [66]:
payout_tool_table = pd.concat(payout_tool_tables, ignore_index=True)
show_confusion_matrix(payout_tool_table, "Payout tool")

In [67]:
payout_tool_table[~payout_tool_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,llm_cost,rationale,run
4,CLM-1205,approve,approve,1021.14,1103.14,False,82.00,0.000494,Approve under Section 3: the carrier confirms ...,1
8,CLM-1209,approve,decline,4180.15,0.00,False,1500.00,0.000822,The incident report identifies the Urubamba ac...,1
17,CLM-1218,approve,decline,4427.95,0.00,False,1500.00,0.000800,Approve under section 2: the carrier confirms ...,1
23,CLM-1224,refer,approve,0.00,1227.95,False,1227.95,0.000357,"Approve the theft of the smartphone, wallet an...",1


In [68]:
print("The baseline got either decision or amount wrong for:", len(baseline_table[~baseline_table["right"]]) )
print("The payout tool version got either decision or amount wrong for:", len(payout_tool_table[~payout_tool_table["right"]]) )


The baseline got either decision or amount wrong for: 11
The payout tool version got either decision or amount wrong for: 4


Open the traces for these claims in Langfuse to see the lines the agent passed to the tool and the calculation that came back.

### Comparing the experiments

In [69]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


pd.DataFrame([average_scores(baseline_table), average_scores(payout_tool_table)], index=["Baseline", "Payout tool"]).round(3)

,decision_accuracy,accuracy,precision,recall,kappa,cost_of_mistakes,straight_through,llm_cost
Baseline,0.84,0.56,1.000,0.714,0.751,4008.42,0.64,0.007
Payout tool,0.88,0.84,0.923,0.857,0.802,4309.95,0.80,0.012


In Langfuse, open **Datasets** and then `harder-claims`. Select the runs to compare them side by side, claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: on Week 1 Day 4 we wrote a concise version of the policy wording, in <code>data/policy_wording_short.md</code>. Run a third experiment with the concise wording and the payout tool, and compare it with the full wording in Langfuse. Does it keep the accuracy? Is it cheaper or faster?<br/><br/>
            Then change <code>WRONGFUL_DECLINE</code> and score the runs again. Does it change which wording you would choose?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Experiment 3: concise wording

Same payout tool as experiment 2, but the system prompt uses `data/policy_wording_short.md` instead of the full wording. Own names, so experiment 2's `payout_tool_table` is left alone.

In [70]:
DATA = Path("../data")

short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")


print(f"The short version is {len(short_wording.split()):,} words")

The short version is 808 words


In [71]:
concise_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{short_wording}\n",
)


def concise_task(*, item, **kwargs):
    """Run the payout-tool agent with the short policy wording."""
    return decide(concise_agent, item.input)

In [72]:
concise_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    concise_run = langfuse.run_experiment(
        name="Payout tool + short wording",
        description="The payout tool, with the concise policy wording",
        data=items,
        task=concise_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(concise_run.format())
    concise_tables.append(score_table(concise_run.item_results).assign(run=run))

CLM-1225: refer $0.00
CLM-1224: refer $0.00
CLM-1223: refer $0.00
CLM-1222: refer $0.00
CLM-1221: refer $0.00
CLM-1220: decline $0.00
CLM-1219: decline $0.00
CLM-1218: approve $4,427.95
CLM-1217: approve $1,335.90
CLM-1216: approve $2,005.90
CLM-1215: approve $1,415.78
CLM-1214: approve $622.60
CLM-1213: decline $0.00
CLM-1212: refer $0.00
CLM-1211: refer $0.00
CLM-1210: decline $0.00
CLM-1209: approve $4,180.15
CLM-1208: approve $3,668.80
CLM-1207: approve $1,189.73
CLM-1206: approve $1,450.01
CLM-1205: approve $1,021.14
CLM-1204: approve $4,787.25
CLM-1203: approve $2,910.44
CLM-1202: approve $2,048.44
CLM-1201: approve $553.28
Individual Results: Hidden (25 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool + short wording
📋 Run name: Payout tool + short wording - 2026-10-22T21:26:51.013640Z - The payout tool, with the concise policy wording
25 items
Evaluations:
  • mistake_cost
  • right

Average Scores

In [73]:
concise_wording_table = pd.concat(concise_tables, ignore_index=True)
show_confusion_matrix(concise_wording_table, "Short wording")

In [74]:
concise_wording_table[~concise_wording_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,llm_cost,rationale,run
6,CLM-1207,approve,approve,1306.72,1189.73,False,116.99,0.000611,Approve the covered stolen baggage items: thef...,1
15,CLM-1216,approve,approve,1677.90,2005.90,False,328.00,0.000512,Approve the covered theft: the police report d...,1
20,CLM-1221,decline,refer,0.00,0.00,False,50.00,0.000367,Refer for review under the Trip Delay section:...,1


In [75]:
print("The baseline got either decision or amount wrong for:", len(baseline_table[~baseline_table["right"]]))
print("The payout tool version got either decision or amount wrong for:", len(payout_tool_table[~payout_tool_table["right"]]))
print("The concise wording version got either decision or amount wrong for:", len(concise_wording_table[~concise_wording_table["right"]]))

The baseline got either decision or amount wrong for: 11
The payout tool version got either decision or amount wrong for: 4
The concise wording version got either decision or amount wrong for: 3


In [76]:
pd.DataFrame(
    [
        average_scores(baseline_table),
        average_scores(payout_tool_table),
        average_scores(concise_wording_table),
    ],
    index=["Baseline", "Payout tool", "Short wording"],
).round(3)

,decision_accuracy,accuracy,precision,recall,kappa,cost_of_mistakes,straight_through,llm_cost
Baseline,0.84,0.56,1.000,0.714,0.751,4008.42,0.64,0.007
Payout tool,0.88,0.84,0.923,0.857,0.802,4309.95,0.80,0.012
Short wording,0.96,0.88,1.000,1.000,0.932,494.99,0.72,0.011


In [77]:

def wrongful_declines(table):
    """Agent declined a claim the handler did not."""
    return int(((table["agent"] == "decline") & (table["handler"] != "decline")).sum())


def cost_with_decline(table, decline_cost):
    """Sum mistake_cost as if WRONGFUL_DECLINE were decline_cost. No new model calls."""
    total = 0.0
    for row in table.itertuples():
        answer = {"decision": row.agent, "payout": row.agent_payout}
        expected = {"decision": row.handler, "payout": row.handler_payout}
        if answer["decision"] == expected["decision"]:
            total += abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
        elif answer["decision"] == "decline":
            total += decline_cost
        elif answer["decision"] == "approve":
            total += answer["payout"]
        else:
            total += NEEDLESS_REFERRAL
    return total


runs = {
    "Baseline": baseline_table,
    "Payout tool": payout_tool_table,
    "Short wording": concise_wording_table,
}
print("Wrongful declines in this kernel:")
for name, table in runs.items():
    print(f"  {name}: {wrongful_declines(table)}")

pd.DataFrame(
    {
        "wrongful_declines": {name: wrongful_declines(table) for name, table in runs.items()},
        "cost_at_1500": {name: cost_with_decline(table, 1500) for name, table in runs.items()},
        "cost_at_5000": {name: cost_with_decline(table, 5000) for name, table in runs.items()},
    }
).round(2)

Wrongful declines in this kernel:
  Baseline: 1
  Payout tool: 2
  Short wording: 0


,wrongful_declines,cost_at_1500,cost_at_5000
Baseline,1,4008.42,7508.42
Payout tool,2,4309.95,11309.95
Short wording,0,494.99,494.99


## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [31]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")